# Hierarchical BART panel forecast

This example forecasts weekly outbound ridership for the busiest BART stations. The model uses the named `series` dimension throughout and partially pools station intercepts, trend scales, seasonal coefficients, and observation scales.

In [ ]:
import logging

import arviz as az
import matplotlib.pyplot as plt
import pymc as pm
import pytensor.tensor as pt
import xarray as xr

from pymc_forecast import Forecaster, ForecastingModel, Horizon, evaluate_forecast, fourier_features
from pymc_forecast.data import FUTURE_DIM, TIME_DIM
from pymc_forecast.datasets import load_bart_weekly_by_origin

az.style.use("arviz-darkgrid")
logging.getLogger("pymc").setLevel(logging.ERROR)
SEED = 42

## Labeled panel data

The loader aggregates each hourly origin-destination shard before retaining the busiest origins, so the notebook never materializes the full three-dimensional panel.

In [ ]:
y = load_bart_weekly_by_origin(num_series=6)
duration = y.sizes[TIME_DIM]
test_window = 26
split = duration - test_window
y_train = y.isel({TIME_DIM: slice(None, split)})
y_test = y.isel({TIME_DIM: slice(split, None)}).rename({TIME_DIM: FUTURE_DIM})

seasonality = fourier_features(duration, period=365.25 / 7, num_terms=5)
seasonality = seasonality.rename({"fourier": "covariate"})
seasonality_train = seasonality.isel({TIME_DIM: slice(None, split)})
print(y.sizes)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for station in y["series"].values:
    ax.plot(y[TIME_DIM], y.sel(series=station), lw=1, label=str(station))
ax.axvline(split, color="black", ls="--", label="train/test split")
ax.set(xlabel="week", ylabel="log(1 + rides)", title="Weekly outbound BART ridership")
ax.legend(ncol=2)
plt.show()

## A dims-first hierarchical model

Every station-specific variable carries `dims="series"`; the time-varying innovation has dims `("time", "series")`. Hyperpriors pool information across stations without flattening either dimension.

In [ ]:
class HierarchicalBART(ForecastingModel):
    def __init__(self, center: float):
        self.center = center

    def model(self, h: Horizon, covariates: xr.DataArray) -> None:
        global_level = pm.Normal("global_level", self.center, 1.0)
        level_scale = pm.HalfNormal("level_scale", 0.5)
        level_offset = pm.Normal("level_offset", 0.0, 1.0, dims="series")
        intercept = global_level + level_scale * level_offset

        global_drift_scale = pm.HalfNormal("global_drift_scale", 0.03)
        drift_scale = pm.HalfNormal("drift_scale", global_drift_scale, dims="series")
        drift_raw = self.time_series(
            "drift_raw",
            lambda name, dims: pm.Normal(name, 0.0, 1.0, dims=dims),
            dims=("series",),
        )
        trend = pt.cumsum(drift_raw * drift_scale, axis=0)

        seasonal_mean = pm.Normal("seasonal_mean", 0.0, 0.1, dims="covariate")
        seasonal_scale = pm.HalfNormal("seasonal_scale", 0.1, dims="covariate")
        seasonal_weight = pm.Normal(
            "seasonal_weight",
            seasonal_mean,
            seasonal_scale,
            dims=("series", "covariate"),
        )
        seasonal = pt.dot(covariates.values, seasonal_weight.T)

        sigma_global = pm.HalfNormal("sigma_global", 0.2)
        sigma = pm.HalfNormal("sigma", sigma_global, dims="series")
        nu = pm.Gamma("nu", alpha=10, beta=2)
        self.predict(
            lambda name, mu, dims, observed: pm.StudentT(
                name, nu=nu, mu=mu, sigma=sigma, dims=dims, observed=observed
            ),
            intercept + trend + seasonal,
        )


model = HierarchicalBART(float(y_train.mean()))

## Fit once, forecast every station

ADVI fits the joint model. Forecasting extends the Fourier design matrix, replays the shared posterior, and samples the separate `drift_raw_future` innovations.

In [ ]:
forecaster = Forecaster(
    model,
    y_train,
    seasonality_train,
    optimizer=0.005,
    num_steps=8_000,
    random_seed=SEED,
)
idata = forecaster.forecast(seasonality, num_samples=500, random_seed=SEED)
forecast = idata["predictions"]["forecast"]
scores = evaluate_forecast(forecast, y_test)
print(scores)
print(forecast.sizes)

In [ ]:
station = str(y["series"].values[0])
station_samples = forecast.sel(series=station).stack(sample=("chain", "draw"))
mean = station_samples.mean("sample")
lo, hi = station_samples.quantile([0.05, 0.95], dim="sample").values

fig, ax = plt.subplots(figsize=(11, 5))
history = y.sel(series=station).isel({TIME_DIM: slice(-78, None)})
ax.plot(history[TIME_DIM], history, color="black", label="observed")
ax.plot(forecast[FUTURE_DIM], mean, color="C1", label="forecast mean")
ax.fill_between(forecast[FUTURE_DIM], lo, hi, color="C1", alpha=0.25, label="90% interval")
ax.axvline(split, color="gray", ls="--")
ax.set(title=f"Hierarchical forecast: {station}", xlabel="week", ylabel="log(1 + rides)")
ax.legend()
plt.show()